In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
GPUs: 2


In [3]:
!pip install -q roboflow torchmetrics pycocotools albumentations

In [4]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key="XslFaDHQ2fu2uFnuAF5K")
project = rf.workspace("image-analysis-project-rfnyq").project("human-coastal-zmajf")
version = project.version(1)
dataset = version.download("yolov8")

LOCAL_DATASET_PATH = dataset.location
DATASET_PATH = LOCAL_DATASET_PATH
DATASET_DRIVE_PATH = LOCAL_DATASET_PATH
print("Dataset downloaded to:", LOCAL_DATASET_PATH)
print(os.listdir(LOCAL_DATASET_PATH))

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to human-coastal-1 in yolov8:: 100%|██████████| 2267/2267 [00:01<00:00, 2086.57it/s]


Dataset downloaded to: /kaggle/working/human-coastal-1
['valid', 'test', 'data.yaml', 'README.roboflow.txt', 'train', 'README.dataset.txt']


In [5]:
import yaml

with open(os.path.join(LOCAL_DATASET_PATH, "data.yaml")) as f:
    data_yaml = yaml.safe_load(f)

print(data_yaml)
CLASS_NAMES_RAW = {i: name for i, name in enumerate(data_yaml['names'])}
print("Classes:", CLASS_NAMES_RAW)

{'names': ['0', '1'], 'nc': 2, 'roboflow': {'license': 'CC BY 4.0', 'project': 'human-coastal-zmajf', 'url': 'https://universe.roboflow.com/image-analysis-project-rfnyq/human-coastal-zmajf/dataset/1', 'version': 1, 'workspace': 'image-analysis-project-rfnyq'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}
Classes: {0: '0', 1: '1'}


In [6]:
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as F

RAW_TO_MODEL = {0: 1, 1: 2}   # torchvision reserves label 0 for background
MODEL_TO_RAW = {1: 0, 2: 1}
NUM_CLASSES_WITH_BG = 3

class YoloDetectionDataset(Dataset):
    def __init__(self, split_dir, transforms=None):
        self.images_dir = os.path.join(split_dir, "images")
        self.labels_dir = os.path.join(split_dir, "labels")
        self.transforms = transforms
        self.image_files = sorted([
            f for f in os.listdir(self.images_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        label_path = os.path.join(self.labels_dir, os.path.splitext(img_name)[0] + ".txt")

        img = Image.open(img_path).convert("RGB")
        img_w, img_h = img.size

        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, xc, yc, w, h = parts
                    cls = int(cls)
                    xc, yc, w, h = float(xc), float(yc), float(w), float(h)

                    x1 = (xc - w / 2) * img_w
                    y1 = (yc - h / 2) * img_h
                    x2 = (xc + w / 2) * img_w
                    y2 = (yc + h / 2) * img_h

                    if x2 <= x1 or y2 <= y1:
                        continue

                    boxes.append([x1, y1, x2, y2])
                    labels.append(RAW_TO_MODEL[cls])

        boxes = torch.as_tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) else torch.zeros((0,)),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }

        img = F.to_tensor(img)
        if self.transforms:
            img, target = self.transforms(img, target)
        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

In [7]:
from torch.utils.data import DataLoader

TRAIN_DIR = os.path.join(LOCAL_DATASET_PATH, "train")
VALID_DIR = os.path.join(LOCAL_DATASET_PATH, "valid")
TEST_DIR  = os.path.join(LOCAL_DATASET_PATH, "test")

train_dataset = YoloDetectionDataset(TRAIN_DIR)
valid_dataset = YoloDetectionDataset(VALID_DIR)
test_dataset  = YoloDetectionDataset(TEST_DIR)

print("Train:", len(train_dataset), " Valid:", len(valid_dataset), " Test:", len(test_dataset))

N_GPUS = max(1, torch.cuda.device_count())
BATCH_SIZE = 4 * N_GPUS          # 4 images per GPU
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, collate_fn=collate_fn)

# sanity check
img, target = train_dataset[0]
print(img.shape, target)

Train: 788  Valid: 229  Test: 114
torch.Size([3, 3024, 4032]) {'boxes': tensor([[3971.0000, 1044.0000, 3992.0000, 1083.0000],
        [1368.0000, 1140.0000, 1393.0000, 1171.0000],
        [3321.0000, 1213.0000, 3345.0000, 1248.0000],
        [2315.0000, 1219.0000, 2335.0000, 1244.0000],
        [3802.0000, 1220.0000, 3827.0000, 1248.0000],
        [2602.0000, 1225.0000, 2634.0000, 1262.0000],
        [2912.0000, 1217.0000, 2942.0000, 1274.0000],
        [3662.0000, 1228.7300, 3695.0000, 1268.7300],
        [2828.0000, 1226.0000, 2865.0000, 1269.0000],
        [2746.0000, 1232.0000, 2776.0000, 1272.0000],
        [2746.0000, 1232.0000, 2777.0000, 1272.0000],
        [3203.0000, 1234.3300, 3229.0000, 1274.3300],
        [3861.2700, 1240.4700, 3895.2700, 1287.4700],
        [2438.0000, 1246.4600, 2466.0000, 1284.4600],
        [2439.0000, 1248.0000, 2466.0000, 1286.0000],
        [2678.0000, 1271.0000, 2713.0000, 1370.0000],
        [2323.0000, 1292.0000, 2355.0000, 1372.0000],
        [3

In [8]:
import torch.nn as nn
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

class DetectionDataParallel(nn.DataParallel):
    """DataParallel that works with torchvision detection models (lists of images/targets)."""
    def scatter(self, inputs, kwargs, device_ids):
        images, targets = inputs
        per = (len(images) + len(device_ids) - 1) // len(device_ids)
        chunks = []
        for i, dev in enumerate(device_ids):
            imgs = images[i * per:(i + 1) * per]
            if not imgs:
                break
            tgts = targets[i * per:(i + 1) * per]
            chunks.append((
                [im.to(dev, non_blocking=True) for im in imgs],
                [{k: v.to(dev, non_blocking=True) for k, v in t.items()} for t in tgts],
            ))
        return tuple(chunks), tuple({} for _ in chunks)

    def gather(self, outputs, output_device):
        # average each loss component across GPUs
        return {k: torch.stack([o[k].to(output_device) for o in outputs]).mean() for k in outputs[0]}

def get_model(num_classes=NUM_CLASSES_WITH_BG):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = get_model()
model.to(device)
parallel_model = DetectionDataParallel(model)   # uses both GPUs for training
print("Training on", torch.cuda.device_count(), "GPU(s)")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:00<00:00, 221MB/s]  


Training on 2 GPU(s)


In [3]:
import time

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)

NUM_EPOCHS = 150
history = {"train_loss": [], "val_loss": [], "loss_classifier": [], "loss_box_reg": [], "loss_objectness": [], "loss_rpn_box_reg": []}

def evaluate_loss(parallel_model, data_loader):
    parallel_model.train()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for images, targets in data_loader:
            loss_dict = parallel_model(images, targets)
            total_loss += sum(loss for loss in loss_dict.values()).item()
            n += 1
    return total_loss / max(n, 1)

for epoch in range(NUM_EPOCHS):
    parallel_model.train()
    epoch_loss = 0.0
    epoch_components = {"loss_classifier": 0.0, "loss_box_reg": 0.0, "loss_objectness": 0.0, "loss_rpn_box_reg": 0.0}
    t0 = time.time()

    for images, targets in train_loader:
        loss_dict = parallel_model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        for k in epoch_components:
            if k in loss_dict:
                epoch_components[k] += loss_dict[k].item()

    lr_scheduler.step()
    avg_train_loss = epoch_loss / len(train_loader)
    avg_val_loss = evaluate_loss(parallel_model, valid_loader)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    for k in epoch_components:
        history[k].append(epoch_components[k] / len(train_loader))

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f} time={time.time()-t0:.1f}s")

CKPT_PATH = os.path.join(DATASET_DRIVE_PATH, "fasterrcnn_beach_human.pth")
torch.save(model.state_dict(), CKPT_PATH)   # raw model -> no "module." prefix
print("Saved:", CKPT_PATH)

NameError: name 'model' is not defined

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(history["train_loss"], label="Train Loss", marker='o')
plt.plot(history["val_loss"], label="Validation Loss", marker='o')
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Faster R-CNN Training vs Validation Loss")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8,5))
for k in ["loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]:
    plt.plot(history[k], label=k, marker='o')
plt.xlabel("Epoch"); plt.ylabel("Loss component"); plt.title("Loss Component Breakdown per Epoch")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        preds = [{k: v.cpu() for k, v in o.items()} for o in outputs]
        tgts  = [{k: v.cpu() for k, v in t.items()} for t in targets]
        all_preds.extend(preds)
        all_targets.extend(tgts)

print("Collected predictions for", len(all_preds), "test images")

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision(class_metrics=True)
metric.update(all_preds, all_targets)
results = metric.compute()

print("mAP@[0.5:0.95]:", round(results['map'].item(), 4))
print("mAP@0.5:        ", round(results['map_50'].item(), 4))
print("mAP@0.75:       ", round(results['map_75'].item(), 4))
print("mAR (100 dets): ", round(results['mar_100'].item(), 4))

if 'map_per_class' in results and results['map_per_class'].numel() > 0:
    classes = results.get('classes', torch.tensor([1, 2])).tolist()
    for model_label, ap in zip(classes, results['map_per_class'].tolist()):
        raw = MODEL_TO_RAW.get(model_label, model_label)
        print(f"  AP class {raw} ({CLASS_NAMES_RAW.get(raw, raw)}): {round(ap,4)}")

In [ ]:
import numpy as np

def box_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

IOU_THRESH = 0.5
SCORE_THRESH = 0.5

y_true, y_pred, y_scores = [], [], []
total_gt, total_matched, total_fp = 0, 0, 0

for pred, tgt in zip(all_preds, all_targets):
    matched_gt = set()
    p_boxes, p_labels, p_scores = pred['boxes'].numpy(), pred['labels'].numpy(), pred['scores'].numpy()
    g_boxes, g_labels = tgt['boxes'].numpy(), tgt['labels'].numpy()
    total_gt += len(g_boxes)

    keep = p_scores > SCORE_THRESH
    p_boxes, p_labels, p_scores = p_boxes[keep], p_labels[keep], p_scores[keep]

    for pb, pl, ps in zip(p_boxes, p_labels, p_scores):
        best_iou, best_j = 0, -1
        for j, gb in enumerate(g_boxes):
            if j in matched_gt:
                continue
            iou = box_iou(pb, gb)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= IOU_THRESH:
            matched_gt.add(best_j)
            total_matched += 1
            y_true.append(MODEL_TO_RAW.get(int(g_labels[best_j]), int(g_labels[best_j])))
            y_pred.append(MODEL_TO_RAW.get(int(pl), int(pl)))
            y_scores.append(ps)
        else:
            total_fp += 1  # unmatched prediction = false positive

y_true, y_pred, y_scores = np.array(y_true), np.array(y_pred), np.array(y_scores)
print(f"Total GT boxes: {total_gt} | Matched (TP): {total_matched} | Unmatched preds (FP): {total_fp}")

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)

detection_accuracy = total_matched / total_gt if total_gt > 0 else 0
classification_accuracy = accuracy_score(y_true, y_pred)  # accuracy of 0-vs-1 label among matched detections

precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

print(f"Detection recall (TP / total GT boxes): {detection_accuracy:.4f}")
print(f"Classification accuracy (0 vs 1, on matched boxes): {classification_accuracy:.4f}")
print(f"Precision (macro): {precision_macro:.4f}")
print(f"Recall (macro):    {recall_macro:.4f}")
print(f"F1-score (macro):  {f1_macro:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=[CLASS_NAMES_RAW.get(0,"class0"), CLASS_NAMES_RAW.get(1,"class1")], zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[f"0: {CLASS_NAMES_RAW.get(0,'class0')}", f"1: {CLASS_NAMES_RAW.get(1,'class1')}"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Faster R-CNN")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(7,6))
for cls, name in CLASS_NAMES_RAW.items():
    binary_true = (y_true == cls).astype(int)
    binary_scores = np.where(y_pred == cls, y_scores, 0)
    if binary_true.sum() == 0:
        continue
    precision, recall, _ = precision_recall_curve(binary_true, binary_scores)
    ap = average_precision_score(binary_true, binary_scores)
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")

plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall Curve per Class")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
metrics_summary = {
    "Precision": precision_macro,
    "Recall": recall_macro,
    "F1-score": f1_macro,
    "mAP@0.5": results['map_50'].item(),
    "mAP@[.5:.95]": results['map'].item(),
}

plt.figure(figsize=(8,5))
bars = plt.bar(metrics_summary.keys(), metrics_summary.values(), color=['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974'])
plt.ylim(0, 1.0)
plt.ylabel("Score")
plt.title("Faster R-CNN — Summary Metrics")
for bar, val in zip(bars, metrics_summary.values()):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}", ha='center')
plt.grid(axis='y', alpha=0.3)
plt.show()

print(metrics_summary)

In [ ]:
import matplotlib.patches as patches

def show_predictions(dataset, model, num_images=4, score_thresh=0.6):
    model.eval()
    fig, axes = plt.subplots(1, num_images, figsize=(5*num_images, 5))
    for i in range(num_images):
        img, target = dataset[i]
        with torch.no_grad():
            pred = model([img.to(device)])[0]
        img_np = img.permute(1,2,0).numpy()
        ax = axes[i] if num_images > 1 else axes
        ax.imshow(img_np)
        for box, label, score in zip(pred['boxes'], pred['labels'], pred['scores']):
            if score < score_thresh:
                continue
            raw_label = MODEL_TO_RAW.get(int(label), int(label))
            x1,y1,x2,y2 = box.cpu().numpy()
            rect = patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-5, f"{raw_label}:{score:.2f}", color='lime', fontsize=10, weight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_predictions(test_dataset, model, num_images=4)

In [ ]:
model.eval()
sample_imgs = [test_dataset[i][0].to(device) for i in range(min(20, len(test_dataset)))]

# warm-up
with torch.no_grad():
    _ = model([sample_imgs[0]])

times = []
with torch.no_grad():
    for img in sample_imgs:
        t0 = time.time()
        _ = model([img])
        times.append(time.time() - t0)

avg_inference_time = np.mean(times)
fps = 1 / avg_inference_time
print(f"Avg inference time per image: {avg_inference_time*1000:.1f} ms")
print(f"Approx FPS: {fps:.2f}")

In [ ]:
from torchvision.transforms import functional as TF

TEST_IMAGE_DIR = os.path.join(LOCAL_DATASET_PATH, "test", "images")
CONFIDENCE_THRESHOLD = 0.60
NUM_IMAGES = 6

model.eval()

image_files = sorted([
    os.path.join(TEST_IMAGE_DIR, f)
    for f in os.listdir(TEST_IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])
print("Total test images:", len(image_files))

for image_path in image_files[:NUM_IMAGES]:
    image = Image.open(image_path).convert("RGB")
    image_tensor = TF.to_tensor(image).to(device)

    with torch.no_grad():
        output = model([image_tensor])[0]

    boxes = output["boxes"].cpu()
    labels = output["labels"].cpu()
    scores = output["scores"].cpu()

    keep = scores >= CONFIDENCE_THRESHOLD
    boxes, labels, scores = boxes[keep], labels[keep], scores[keep]

    land_count = int((labels == 1).sum())
    water_count = int((labels == 2).sum())
    total_count = land_count + water_count

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)

    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box.tolist()

        if label.item() == 1:
            class_name = "Person on Land"
        elif label.item() == 2:
            class_name = "Person on Water"
        else:
            continue

        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 5, 0), f"{class_name} ({score:.2f})", fontsize=9, backgroundcolor="white")

    count_text = (
        f"PERSON ON LAND  : {land_count}\n"
        f"PERSON ON WATER : {water_count}\n"
        f"TOTAL PEOPLE    : {total_count}"
    )
    ax.text(0.02, 0.03, count_text, transform=ax.transAxes, fontsize=14,
            fontweight="bold", verticalalignment="bottom", backgroundcolor="white")

    ax.set_title(f"Predictions - {os.path.basename(image_path)}")
    ax.axis("off")
    plt.show()

    print(f"{os.path.basename(image_path)} | Land = {land_count} | Water = {water_count} | Total = {total_count}")

In [ ]:
# 0 = LAND, 1 = WATER
CONFIDENCE_THRESHOLD = 0.60
model.eval()

print("\n📷 Enter the path of an image available in Kaggle:")
image_name = input("Image path: ").strip()

if not os.path.exists(image_name):
    print("❌ Image not found:", image_name)

else:
    image = Image.open(image_name).convert("RGB")
    image_tensor = TF.to_tensor(image).to(device)

    with torch.no_grad():
        prediction = model([image_tensor])[0]

    boxes = prediction["boxes"].cpu()
    labels = prediction["labels"].cpu()
    scores = prediction["scores"].cpu()

    keep = scores >= CONFIDENCE_THRESHOLD
    boxes, labels, scores = boxes[keep], labels[keep], scores[keep]

    land_count = int((labels == 1).sum())    # model label 1 -> class 0 -> Land
    water_count = int((labels == 2).sum())   # model label 2 -> class 1 -> Water
    total_count = land_count + water_count

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)

    for box, label, score in zip(boxes, labels, scores):
        raw_label = MODEL_TO_RAW.get(int(label), int(label))
        x1, y1, x2, y2 = box.numpy()

        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 5, 5), f"{raw_label}:{score:.2f}", color="lime",
                fontsize=9, weight="bold", verticalalignment="bottom")

    count_text = f"Land (0): {land_count}    Water (1): {water_count}    Total: {total_count}"
    ax.text(0.02, 0.98, count_text, transform=ax.transAxes, fontsize=11, weight="bold",
            color="white", verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="black", alpha=0.65, edgecolor="none"))

    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("\n========== LIVE COUNT ==========")
    print(f"People on Land  (0): {land_count}")
    print(f"People on Water (1): {water_count}")
    print(f"Total People       : {total_count}")
    print("================================")

In [ ]:
!pip -q install ultralytics

import shutil
from ultralytics import YOLO

DATASET_PATH = LOCAL_DATASET_PATH

SPLITS = {
    "train": (os.path.join(DATASET_PATH, "train", "images"), os.path.join(DATASET_PATH, "train", "labels")),
    "valid": (os.path.join(DATASET_PATH, "valid", "images"), os.path.join(DATASET_PATH, "valid", "labels")),
    "test":  (os.path.join(DATASET_PATH, "test",  "images"), os.path.join(DATASET_PATH, "test",  "labels")),
}

# Backup of original labels
BACKUP_PATH = os.path.join(DATASET_PATH, "labels_backup_before_human_cleaning")
os.makedirs(BACKUP_PATH, exist_ok=True)
print("Creating backup of original labels...")

for split, (image_dir, label_dir) in SPLITS.items():
    if not os.path.exists(label_dir):
        print(f"⚠️ Label folder not found: {label_dir}")
        continue
    backup_split = os.path.join(BACKUP_PATH, split)
    os.makedirs(backup_split, exist_ok=True)
    for filename in os.listdir(label_dir):
        if filename.endswith(".txt"):
            shutil.copy2(os.path.join(label_dir, filename), os.path.join(backup_split, filename))

print("✅ Original labels backed up to:", BACKUP_PATH)

# Pretrained person detector (COCO class 0 = person)
print("\nLoading pretrained person detector...")
person_model = YOLO("yolov8m.pt")
print("✅ Person detector loaded")

PERSON_CONFIDENCE = 0.20
IOU_THRESHOLD = 0.20      # lower = keeps more boxes, higher = more aggressive cleaning
USE_CENTER_TEST = True    # also keep a box if its center lies inside a detected person

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - intersection

    if union <= 0:
        return 0.0
    return intersection / union

def center_inside(box, person_box):
    cx = (box[0] + box[2]) / 2
    cy = (box[1] + box[3]) / 2
    return person_box[0] <= cx <= person_box[2] and person_box[1] <= cy <= person_box[3]

def clean_label_file(image_path, label_path):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    if not os.path.exists(label_path):
        return 0, 0

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    if len(lines) == 0:
        return 0, 0

    original_count = 0
    kept_count = 0
    annotation_data = []

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            continue
        try:
            class_id = int(parts[0])
            xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        except:
            continue

        if class_id not in [0, 1]:
            continue

        x_center, y_center = xc * width, yc * height
        box_width, box_height = bw * width, bh * height

        x1 = x_center - box_width / 2
        y1 = y_center - box_height / 2
        x2 = x_center + box_width / 2
        y2 = y_center + box_height / 2

        annotation_data.append({"class_id": class_id, "yolo_line": line, "box": [x1, y1, x2, y2]})
        original_count += 1

    results_p = person_model.predict(source=image, conf=PERSON_CONFIDENCE, classes=[0], verbose=False)
    result = results_p[0]

    detected_person_boxes = []
    if result.boxes is not None:
        for box, conf in zip(result.boxes.xyxy.cpu().numpy(), result.boxes.conf.cpu().numpy()):
            if conf >= PERSON_CONFIDENCE:
                detected_person_boxes.append(box.tolist())

    kept_lines = []
    for annotation in annotation_data:
        annotation_box = annotation["box"]
        is_human = False

        for person_box in detected_person_boxes:
            if calculate_iou(annotation_box, person_box) >= IOU_THRESHOLD:
                is_human = True
                break
            if USE_CENTER_TEST and center_inside(annotation_box, person_box):
                is_human = True
                break

        if is_human:
            kept_lines.append(annotation["yolo_line"])
            kept_count += 1

    with open(label_path, "w") as f:
        if kept_lines:
            f.write("\n".join(kept_lines) + "\n")
        else:
            f.write("")   # empty label file = no humans

    return original_count, kept_count

total_original = 0
total_kept = 0
total_removed = 0

for split, (image_dir, label_dir) in SPLITS.items():
    print("\n" + "=" * 60)
    print(f"PROCESSING {split.upper()}")
    print("=" * 60)

    if not os.path.exists(image_dir):
        print("⚠️ Image folder not found:", image_dir)
        continue

    os.makedirs(label_dir, exist_ok=True)

    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ])

    split_original = 0
    split_kept = 0

    for index, filename in enumerate(image_files):
        image_path = os.path.join(image_dir, filename)
        label_path = os.path.join(label_dir, os.path.splitext(filename)[0] + ".txt")

        original, kept = clean_label_file(image_path, label_path)
        split_original += original
        split_kept += kept

        if original > kept:
            print(f"[{index+1}/{len(image_files)}] {filename}: {original} → {kept} (removed {original - kept})")

    split_removed = split_original - split_kept
    total_original += split_original
    total_kept += split_kept
    total_removed += split_removed

    print(f"\n{split.upper()} SUMMARY")
    print("Original boxes :", split_original)
    print("Human boxes    :", split_kept)
    print("Removed boxes  :", split_removed)

print("\n" + "=" * 60)
print("              CLEANING COMPLETE")
print("=" * 60)
print("Original boxes :", total_original)
print("Human boxes    :", total_kept)
print("Removed boxes  :", total_removed)
print("\nNext step: retrain Faster R-CNN using the cleaned labels.")

In [ ]:
# RAW CLASS: 0 = HUMAN ON LAND, 1 = HUMAN ON WATER
TRAIN_IMG_DIR = os.path.join(DATASET_PATH, "train", "images")
TRAIN_LABEL_DIR = os.path.join(DATASET_PATH, "train", "labels")
VAL_IMG_DIR = os.path.join(DATASET_PATH, "valid", "images")
VAL_LABEL_DIR = os.path.join(DATASET_PATH, "valid", "labels")

NEW_MODEL_PATH = os.path.join(DATASET_PATH, "fasterrcnn_beach_human_clean.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

RAW_TO_MODEL = {0: 1, 1: 2}
NUM_CLASSES = 3

class BeachHumanDataset(Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.images = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        filename = self.images[idx]
        image = Image.open(os.path.join(self.image_dir, filename)).convert("RGB")
        width, height = image.size

        label_path = os.path.join(self.label_dir, os.path.splitext(filename)[0] + ".txt")

        boxes, labels = [], []

        # empty label file = no humans
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue

                    raw_class = int(parts[0])
                    if raw_class not in RAW_TO_MODEL:
                        continue

                    xc = float(parts[1]) * width
                    yc = float(parts[2]) * height
                    bw = float(parts[3]) * width
                    bh = float(parts[4]) * height

                    x1 = max(0, min(xc - bw / 2, width))
                    y1 = max(0, min(yc - bh / 2, height))
                    x2 = max(0, min(xc + bw / 2, width))
                    y2 = max(0, min(yc + bh / 2, height))

                    if x2 <= x1 or y2 <= y1:
                        continue

                    boxes.append([x1, y1, x2, y2])
                    labels.append(RAW_TO_MODEL[raw_class])

        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}
        return TF.to_tensor(image), target

N_GPUS = max(1, torch.cuda.device_count())
CLEAN_BATCH_SIZE = 2 * N_GPUS    # 2 images per GPU

train_dataset = BeachHumanDataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR)
val_dataset = BeachHumanDataset(VAL_IMG_DIR, VAL_LABEL_DIR)

train_loader = DataLoader(train_dataset, batch_size=CLEAN_BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CLEAN_BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

# Model (2-GPU)
model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(device)
parallel_model = DetectionDataParallel(model)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

NUM_EPOCHS = 150

for epoch in range(NUM_EPOCHS):
    parallel_model.train()
    total_loss = 0.0

    for images, targets in train_loader:
        loss_dict = parallel_model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    lr_scheduler.step()
    avg_loss = total_loss / max(len(train_loader), 1)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), NEW_MODEL_PATH)

print("\n======================================")
print("✅ CLEAN MODEL TRAINING COMPLETE")
print("======================================")
print("Saved model:", NEW_MODEL_PATH)

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score

# 0 = LAND, 1 = WATER
TEST_IMAGE_DIR = os.path.join(DATASET_PATH, "test", "images")
TEST_LABEL_DIR = os.path.join(DATASET_PATH, "test", "labels")

model.eval()    # the cleaned-label model trained in the previous cell

CONFIDENCE_THRESHOLD = 0.50
IOU_THRESHOLD = 0.50
MODEL_TO_RAW = {1: 0, 2: 1}

requested_images = [
    "1640154603-Wed-Dec-22_16_30_03-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.JRKPYKB0XT4cxKdpZm3b.jpg",
    "1640150102-Wed-Dec-22_15_15_02-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.WSuIM7YlMnrO43HvoTSi.jpg",
    "1640151902-Wed-Dec-22_15_45_02-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.3WehESpX0Qzw7Q91a9NU.jpg",
    "1640149202-Wed-Dec-22_15_00_02-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.ls85n6g3Mrg5KoE9rfKw.jpg",
    "1640155505-Wed-Dec-22_16_45_05-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.osRyOFMWqcUQgxgfWbLo.jpg",
    "1640155503-Wed-Dec-22_16_45_03-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.622aUMZx7BgnBQE6c4Y5.jpg",
    "1640135702-Wed-Dec-22_11_15_02-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.7e47Phitp5U5GjN16rdq.jpg",
    "1640217602-Thu-Dec-23_10_00_02-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.1y4GGYLPw1jTry0zY4fQ.jpg",
    "1639953906-Mon-Dec-20_08_45_06-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.0dGyCV5KVtDrcRPFVPhs.jpg",
    "1640293205-Fri-Dec-24_07_00_05-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.ZaOpUQjD4cHVJMEGcoiO.jpg",
]

# Find available requested images, fill missing slots automatically
available = []
print("\n" + "=" * 70)
print("CHECKING REQUESTED IMAGES")
print("=" * 70)

for filename in requested_images:
    path = os.path.join(TEST_IMAGE_DIR, filename)
    if os.path.exists(path):
        available.append(path)
        print("✅", filename)
    else:
        print("❌ Missing:", filename)

print("\nRequested images available:", len(available), "/ 10")

all_test_images = sorted([
    os.path.join(TEST_IMAGE_DIR, f)
    for f in os.listdir(TEST_IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
])

used_names = set(os.path.basename(p) for p in available)
remaining_images = [p for p in all_test_images if os.path.basename(p) not in used_names]

needed = 10 - len(available)
if needed > 0:
    print("\n" + "=" * 70)
    print("REPLACING MISSING IMAGES")
    print("=" * 70)
    for p in remaining_images[:needed]:
        print("🔄 Replacement:", os.path.basename(p))
        available.append(p)

image_paths = available[:10]

print("\n" + "=" * 70)
print("FINAL 10 IMAGES")
print("=" * 70)
for i, p in enumerate(image_paths, 1):
    print(f"{i}. {os.path.basename(p)}")
print("\nTotal selected:", len(image_paths))

if len(image_paths) < 10:
    raise RuntimeError("The test folder contains fewer than 10 images.")

def read_ground_truth(image_path):
    image_name = os.path.basename(image_path)
    label_path = os.path.join(TEST_LABEL_DIR, os.path.splitext(image_name)[0] + ".txt")

    if not os.path.exists(label_path):
        print("⚠️ No label found:", os.path.basename(label_path))
        return [], []

    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    boxes, labels = [], []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls = int(parts[0])
            if cls not in [0, 1]:
                continue
            xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            boxes.append([(xc - w / 2) * width, (yc - h / 2) * height,
                          (xc + w / 2) * width, (yc + h / 2) * height])
            labels.append(cls)

    return boxes, labels

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - intersection

    if union <= 0:
        return 0
    return intersection / union

results = []
all_gt_labels = []
all_pred_labels = []
total_tp = total_fp = total_fn = 0

print("\n" + "=" * 70)
print("RUNNING MODEL ON 10 IMAGES")
print("=" * 70)

for image_number, image_path in enumerate(image_paths, start=1):
    image = Image.open(image_path).convert("RGB")
    image_tensor = TF.to_tensor(image).to(device)

    with torch.no_grad():
        prediction = model([image_tensor])[0]

    boxes = prediction["boxes"].cpu().numpy()
    labels = prediction["labels"].cpu().numpy()
    scores = prediction["scores"].cpu().numpy()

    keep = scores >= CONFIDENCE_THRESHOLD
    boxes, labels, scores = boxes[keep], labels[keep], scores[keep]

    raw_labels = np.array([MODEL_TO_RAW.get(int(label), int(label)) for label in labels])

    pred_land = int(np.sum(raw_labels == 0))
    pred_water = int(np.sum(raw_labels == 1))
    pred_total = pred_land + pred_water

    gt_boxes, gt_labels = read_ground_truth(image_path)
    gt_land = sum(1 for x in gt_labels if x == 0)
    gt_water = sum(1 for x in gt_labels if x == 1)
    gt_total = len(gt_labels)

    matched_gt = set()
    tp = 0
    fp = 0

    for pred_box, pred_label in zip(boxes, raw_labels):
        best_iou = 0
        best_gt = -1
        for j, gt_box in enumerate(gt_boxes):
            if j in matched_gt:
                continue
            iou = calculate_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt = j

        if best_iou >= IOU_THRESHOLD:
            matched_gt.add(best_gt)
            tp += 1
            all_gt_labels.append(gt_labels[best_gt])
            all_pred_labels.append(int(pred_label))
        else:
            fp += 1

    fn = len(gt_boxes) - len(matched_gt)
    total_tp += tp
    total_fp += fp
    total_fn += fn

    fig, ax = plt.subplots(figsize=(13, 8))
    ax.imshow(image)

    for box, raw_label, score in zip(boxes, raw_labels, scores):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 5, 5), f"{raw_label}:{score:.2f}", color="lime", fontsize=9, weight="bold")

    panel_text = (
        f"Land (0): {pred_land}    Water (1): {pred_water}    Total: {pred_total}\n"
        f"Ground Truth: {gt_total}"
    )
    ax.text(0.02, 0.98, panel_text, transform=ax.transAxes, fontsize=11, weight="bold",
            color="white", verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="black", alpha=0.65, edgecolor="none"))

    ax.set_title(f"Image {image_number}/10", fontsize=13, weight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"{image_number}. {os.path.basename(image_path)}")
    print(f"   Prediction → Land={pred_land}, Water={pred_water}, Total={pred_total}")
    print(f"   Ground Truth → Land={gt_land}, Water={gt_water}, Total={gt_total}")
    print(f"   TP={tp}, FP={fp}, FN={fn}")
    print()

    results.append({
        "Image": image_number,
        "Filename": os.path.basename(image_path),
        "Ground Truth Land": gt_land,
        "Predicted Land": pred_land,
        "Ground Truth Water": gt_water,
        "Predicted Water": pred_water,
        "Ground Truth Total": gt_total,
        "Predicted Total": pred_total,
        "Error": pred_total - gt_total,
        "Absolute Error": abs(pred_total - gt_total),
        "TP": tp, "FP": fp, "FN": fn,
    })

# Results table
df = pd.DataFrame(results)

print("\n" + "=" * 90)
print("FINAL RESULTS FOR 10 IMAGES")
print("=" * 90)
display(df)

# Overall detection metrics
precision_det = total_tp / (total_tp + total_fp) if total_tp + total_fp > 0 else 0
recall_det = total_tp / (total_tp + total_fn) if total_tp + total_fn > 0 else 0
f1_det = 2 * precision_det * recall_det / (precision_det + recall_det) if precision_det + recall_det > 0 else 0

print("\n" + "=" * 60)
print("OVERALL DETECTION METRICS")
print("=" * 60)
print("TP =", total_tp)
print("FP =", total_fp)
print("FN =", total_fn)
print(f"Precision = {precision_det:.4f}")
print(f"Recall    = {recall_det:.4f}")
print(f"F1 Score  = {f1_det:.4f}")

# Counting metrics
print("\n" + "=" * 60)
print("COUNTING PERFORMANCE")
print("=" * 60)
print("Total Ground Truth:", df["Ground Truth Total"].sum())
print("Total Predicted:", df["Predicted Total"].sum())
print("Mean Absolute Error:", round(df["Absolute Error"].mean(), 3))
print("Total Absolute Error:", df["Absolute Error"].sum())

# Confusion matrix
if len(all_gt_labels) > 0:
    y_true = np.array(all_gt_labels)
    y_pred = np.array(all_pred_labels)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print("\n" + "=" * 60)
    print("CONFUSION MATRIX")
    print("=" * 60)
    print(cm)

    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["0: Land", "1: Water"])
    disp.plot(ax=ax, cmap="Blues")
    plt.title("Confusion Matrix — 10 Images")
    plt.tight_layout()
    plt.show()

    print("Classification Precision:", round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4))
    print("Classification Recall:",    round(recall_score(y_true, y_pred, average="macro", zero_division=0), 4))
    print("Classification F1:",        round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))

# Graphs
x = np.arange(1, len(df) + 1)

plt.figure(figsize=(12, 6))
plt.plot(x, df["Ground Truth Total"], marker="o", linewidth=2, label="Ground Truth")
plt.plot(x, df["Predicted Total"], marker="s", linewidth=2, label="Predicted")
plt.xlabel("Image Number"); plt.ylabel("Number of People")
plt.title("Ground Truth vs Predicted People Count")
plt.xticks(x); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 6))
plt.plot(x, df["Ground Truth Land"], marker="o", linewidth=2, label="Ground Truth Land (0)")
plt.plot(x, df["Predicted Land"], marker="s", linewidth=2, label="Predicted Land (0)")
plt.xlabel("Image Number"); plt.ylabel("People")
plt.title("Land People — Ground Truth vs Prediction")
plt.xticks(x); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 6))
plt.plot(x, df["Ground Truth Water"], marker="o", linewidth=2, label="Ground Truth Water (1)")
plt.plot(x, df["Predicted Water"], marker="s", linewidth=2, label="Predicted Water (1)")
plt.xlabel("Image Number"); plt.ylabel("People")
plt.title("Water People — Ground Truth vs Prediction")
plt.xticks(x); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 6))
plt.bar(x, df["Error"])
plt.axhline(0, linewidth=1)
plt.xlabel("Image Number"); plt.ylabel("Prediction Error")
plt.title("People Count Error")
plt.xticks(x); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 6))
plt.plot(x, df["TP"], marker="o", linewidth=2, label="TP")
plt.plot(x, df["FP"], marker="s", linewidth=2, label="FP")
plt.plot(x, df["FN"], marker="^", linewidth=2, label="FN")
plt.xlabel("Image Number"); plt.ylabel("Detections")
plt.title("TP / FP / FN per Image")
plt.xticks(x); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("\n" + "=" * 70)
print("✅ COMPLETE")
print("=" * 70)